# Physics-Informed Hallucination Detection

Notebook 03 established that ~86% of U-Net reconstruction error at 4x is null-space content (hallucination by Bhadra et al.'s definition), and that the PSF predicts hallucination location with r = 0.95.

This notebook builds 8 detectors that exploit these findings, all GT-free at inference. We evaluate at patch level (pixel-level AUROC is inappropriate for diffuse hallucinations) using null-space maps as ground truth following Theunissen et al. (2025).

## Setup

In [ ]:
!pip install fastmri h5py scikit-image pyyaml tqdm -q

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, h5py, json, time, os
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, auc
from scipy.ndimage import gaussian_filter, uniform_filter, laplace
from scipy.stats import pearsonr
from abc import ABC, abstractmethod

mpl.rcParams.update({
    'font.size': 12, 'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'figure.facecolor': 'white',
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Mount Drive and sync data
from google.colab import drive
import subprocess

drive.mount('/content/drive', force_remount=True)

local_val = '/content/data/singlecoil_val'
os.makedirs(local_val, exist_ok=True)
subprocess.run(['rsync', '-a', '--ignore-existing',
                '/content/drive/MyDrive/fastmri/singlecoil_val/', local_val + '/'],
               capture_output=True)
n_files = len(list(Path(local_val).glob('*.h5')))
print(f"Val volumes: {n_files}")

## 1. Utilities, U-Net, and NB03 Functions

Everything needed for reconstruction, null-space decomposition, and the lift procedure that brings U-Net output back to native k-space resolution.

In [ ]:
# ---- Fourier (norm='ortho') ----

def to_kspace(image):
    return torch.fft.fftshift(
        torch.fft.fft2(torch.fft.ifftshift(image, dim=(-2,-1)), norm='ortho'), dim=(-2,-1))

def from_kspace(kspace):
    return torch.fft.fftshift(
        torch.fft.ifft2(torch.fft.ifftshift(kspace, dim=(-2,-1)), norm='ortho'), dim=(-2,-1))

def center_crop(data, shape):
    h, w = data.shape[-2:]
    th, tw = shape
    return data[..., (h-th)//2:(h-th)//2+th, (w-tw)//2:(w-tw)//2+tw]

def center_embed(crop, full_shape, fill=None):
    fH, fW = full_shape
    cH, cW = crop.shape[-2:]
    out = fill.clone() if fill is not None else torch.zeros(*crop.shape[:-2], fH, fW, dtype=crop.dtype)
    out[..., (fH-cH)//2:(fH-cH)//2+cH, (fW-cW)//2:(fW-cW)//2+cW] = crop
    return out

def normalize(x):
    vmin, vmax = x.min(), x.max()
    if vmax - vmin < 1e-10: return torch.zeros_like(x), vmin.item(), vmax.item()
    return (x - vmin) / (vmax - vmin), vmin.item(), vmax.item()


# ---- Masks ----

def create_mask(shape, acceleration=4, mask_type='random', seed=42, center_fraction=0.08):
    rng = np.random.RandomState(seed)
    H, W = shape[-2], shape[-1]
    num_center = max(1, int(round(W * center_fraction)))
    num_total = max(num_center, int(round(W / acceleration)))
    num_outer = num_total - num_center

    mask = np.zeros(W, dtype=np.float32)
    cs = (W - num_center) // 2
    mask[cs:cs+num_center] = 1.0
    outer = np.where(mask == 0)[0]

    if num_outer <= 0 or len(outer) == 0:
        return torch.from_numpy(mask).unsqueeze(0)

    if mask_type == 'random':
        chosen = rng.choice(outer, size=min(num_outer, len(outer)), replace=False)
        mask[chosen] = 1.0
    elif mask_type == 'equispaced':
        step = max(1, len(outer) // num_outer)
        offset = rng.randint(0, step)
        mask[outer[offset::step][:num_outer]] = 1.0
    elif mask_type == 'gaussian':
        sigma = W / 6.0
        probs = np.exp(-0.5 * ((outer - W/2.0) / sigma)**2)
        probs /= probs.sum()
        chosen = rng.choice(outer, size=min(num_outer, len(outer)), replace=False, p=probs)
        mask[chosen] = 1.0
    elif mask_type == 'poisson_disc':
        probs = 1.0 / (1.0 + 3.0 * np.abs(outer - W/2.0) / (W/2.0))
        probs /= probs.sum()
        chosen = rng.choice(outer, size=min(num_outer, len(outer)), replace=False, p=probs)
        mask[chosen] = 1.0

    return torch.from_numpy(mask).unsqueeze(0)


# ---- U-Net (exact NB02 architecture) ----

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), dropout_p=0.05):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs = nn.ModuleList()
        self.dropout = nn.Dropout2d(p=dropout_p)
        in_ch = 1
        for ch in channels:
            self.encoders.append(ConvBlock(in_ch, ch))
            self.pools.append(nn.MaxPool2d(2))
            in_ch = ch
        self.bottleneck = ConvBlock(channels[-1], channels[-1] * 2)
        for ch in reversed(channels):
            self.upconvs.append(nn.ConvTranspose2d(ch * 2, ch, 2, stride=2))
            self.decoders.append(ConvBlock(ch * 2, ch))
        self.final = nn.Conv2d(channels[0], 1, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x)
            x = pool(x); x = self.dropout(x)
        x = self.bottleneck(x)
        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)
            if x.shape != skip.shape:
                x = F.pad(x, [0, skip.shape[3]-x.shape[3], 0, skip.shape[2]-x.shape[2]])
            x = torch.cat([x, skip], dim=1)
            x = dec(x); x = self.dropout(x)
        return self.final(x)


# ---- NB03 lift + decomposition functions ----

def lift_unet_to_native(unet_01, esc_target, ifft_complex_native, gt_complex_native):
    """Lift U-Net [0,1] output to native resolution complex image (uses GT for outer pixels)."""
    H, W = ifft_complex_native.shape
    unet_phys = unet_01 * (esc_target.max() - esc_target.min()) + esc_target.min()
    unet_mag = center_embed(unet_phys, (H, W), fill=gt_complex_native.abs())
    return unet_mag.to(torch.complex64) * torch.exp(1j * torch.angle(ifft_complex_native).to(torch.complex64))

def lift_unet_to_native_inference(unet_01, ifft_mag_320, ifft_complex_native):
    """Inference-compatible lift: uses IFFT for everything (no GT needed)."""
    H, W = ifft_complex_native.shape
    ifft_min, ifft_max = ifft_mag_320.min(), ifft_mag_320.max()
    if ifft_max - ifft_min < 1e-10: ifft_max = ifft_min + 1.0
    unet_phys = unet_01 * (ifft_max - ifft_min) + ifft_min
    unet_mag = center_embed(unet_phys, (H, W), fill=ifft_complex_native.abs())
    return unet_mag.to(torch.complex64) * torch.exp(1j * torch.angle(ifft_complex_native).to(torch.complex64))

def null_space_decomposition_native(recon_native, gt_native, mask, crop_shape=(320, 320)):
    """Bhadra et al. decomposition at native resolution."""
    error = recon_native - gt_native
    error_meas = from_kspace(to_kspace(error) * mask)
    error_null = error - error_meas

    total_e = (error.abs()**2).sum().item()
    null_e = (error_null.abs()**2).sum().item()
    meas_e = (error_meas.abs()**2).sum().item()

    return {
        'null_ratio': null_e / (total_e + 1e-15),
        'meas_ratio': meas_e / (total_e + 1e-15),
        'error_map': center_crop(error.abs(), crop_shape),
        'null_map': center_crop(error_null.abs(), crop_shape),
        'meas_map': center_crop(error_meas.abs(), crop_shape),
    }

def compute_psf_1d(mask):
    m = mask.squeeze().numpy().astype(np.complex128)
    psf = np.abs(np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(m))))
    return psf / (psf.max() + 1e-10) if psf.max() > 0 else psf

@torch.no_grad()
def full_analysis_slice(h5_path, model, device, acceleration=4.0,
                        mask_type='random', seed=42, slice_idx=None):
    """Complete analysis for one slice: IFFT + U-Net + Bhadra decomposition."""
    with h5py.File(h5_path, 'r') as f:
        if slice_idx is None:
            slice_idx = f['kspace'].shape[0] // 2
        kspace = torch.tensor(f['kspace'][slice_idx].copy(), dtype=torch.complex64)
        esc = torch.tensor(f['reconstruction_esc'][slice_idx].copy(), dtype=torch.float32)

    H, W = kspace.shape
    mask = create_mask((H, W), acceleration, mask_type, seed)
    gt_native = from_kspace(kspace)
    ifft_native = from_kspace(kspace * mask)

    ifft_decomp = null_space_decomposition_native(ifft_native, gt_native, mask)

    ifft_mag_320 = center_crop(ifft_native.abs().unsqueeze(0), (320, 320)).squeeze()
    ifft_norm, _, _ = normalize(ifft_mag_320)

    unet_01 = model(ifft_norm.unsqueeze(0).unsqueeze(0).to(device)).squeeze().cpu().clamp(0, 1)

    unet_native = lift_unet_to_native(unet_01, esc, ifft_native, gt_native)
    unet_decomp = null_space_decomposition_native(unet_native, gt_native, mask)

    mse = ((unet_01 * (esc.max()-esc.min()) + esc.min() - esc)**2).mean().item()
    psnr = 10 * np.log10(esc.max().item()**2 / mse) if mse > 0 else 999

    return {
        'ifft_mag_320': ifft_mag_320, 'unet_output_01': unet_01, 'esc_target': esc,
        'ifft_complex_native': ifft_native, 'gt_complex_native': gt_native,
        'ifft_decomp': ifft_decomp, 'unet_decomp': unet_decomp,
        'mask': mask, 'kspace_shape': (H, W), 'psnr': psnr,
        'acceleration': acceleration, 'mask_type': mask_type,
        'slice_idx': slice_idx, 'h5_path': h5_path,
    }


# ---- Load checkpoint ----

try:
    ckpt = torch.load('/content/drive/MyDrive/fastmri/checkpoints/unet_4x_v2_best.pt',
                       map_location=device, weights_only=False)
except OSError:
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    ckpt = torch.load('/content/drive/MyDrive/fastmri/checkpoints/unet_4x_v2_best.pt',
                       map_location=device, weights_only=False)

model = UNet(channels=(32, 64, 128, 256), dropout_p=0.05).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"U-Net: {sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt['epoch']+1}")

# Quick sanity check
h5_path = sorted(Path(local_val).glob('*.h5'))[0]
with h5py.File(str(h5_path), 'r') as f:
    ks = torch.tensor(f['kspace'][f['kspace'].shape[0]//2].copy(), dtype=torch.complex64)
    esc = torch.tensor(f['reconstruction_esc'][f['reconstruction_esc'].shape[0]//2].copy(), dtype=torch.float32)

mask = create_mask(ks.shape, acceleration=4, seed=42)
ifft_320 = center_crop(from_kspace(ks * mask).abs(), (320, 320))
ifft_norm = (ifft_320 - ifft_320.min()) / (ifft_320.max() - ifft_320.min() + 1e-8)
esc_norm = (esc - esc.min()) / (esc.max() - esc.min() + 1e-8)

with torch.no_grad():
    out = model(ifft_norm.unsqueeze(0).unsqueeze(0).to(device)).squeeze().cpu()
l1 = (out.clamp(0,1) - esc_norm).abs().mean().item()
print(f"Sanity check L1: {l1:.4f} (target ~0.04)")

## 2. Detector Definitions

Eight detectors organized by increasing computational cost:

| Detector | Input | Cost | Principle |
|----------|-------|------|-----------|
| IFFT Gradient | IFFT image | Negligible | Sobel + local variance captures aliasing texture |
| |U-Net - IFFT| | Both recons | Negligible | Network additions are mostly null-space content |
| High-Freq Energy | Both recons | Negligible | Laplacian + DoG on network additions (NB03: 13.6x enrichment at high freq) |
| Data Consistency | U-Net + mask | 1 FFT | Null-space projection of reconstruction |
| K-Space Consistency | U-Net + mask | 1 FFT | Error on measured k-space lines projected to image |
| Ref-Free sFRC | Both recons | High | Patch-wise FRC between U-Net and IFFT (Kc et al., 2024) |
| Multi-Mask (8) | k-space + model | 8 forward passes | Std across reconstructions from different masks |
| Learned Risk | IFFT + PSF | 1 small CNN | Trained on null-space maps |

In [ ]:
class HallucinationDetector(ABC):
    @abstractmethod
    def name(self): pass
    @abstractmethod
    def detect(self, analysis): pass


class PSFDetector(HallucinationDetector):
    """IFFT gradient magnitude + local variance. Captures aliasing texture."""
    def name(self): return "IFFT Gradient"
    def detect(self, analysis):
        ifft = analysis['ifft_mag_320']
        img = ifft.unsqueeze(0).unsqueeze(0)
        sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).reshape(1,1,3,3)
        sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).reshape(1,1,3,3)
        grad = torch.sqrt(F.conv2d(img, sx, padding=1)**2 + F.conv2d(img, sy, padding=1)**2 + 1e-8).squeeze()

        ifft_np = ifft.numpy()
        lm = uniform_filter(ifft_np, size=5)
        lv = np.maximum(uniform_filter(ifft_np**2, size=5) - lm**2, 0)

        g_n = grad / (grad.max() + 1e-8)
        v_n = torch.tensor(lv / (lv.max() + 1e-8), dtype=torch.float32)
        return 0.5 * g_n + 0.5 * v_n


class ResidualDetector(HallucinationDetector):
    """|U-Net - IFFT|: what the network changed."""
    def name(self): return "|U-Net - IFFT|"
    def detect(self, analysis):
        ifft = analysis['ifft_mag_320']
        unet = analysis['unet_output_01'] * (ifft.max() - ifft.min()) + ifft.min()
        return (unet - ifft).abs()


class DataConsistencyDetector(HallucinationDetector):
    """P_null(U-Net): null-space content of reconstruction."""
    def name(self): return "Data Consistency"
    def detect(self, analysis):
        unet_native = lift_unet_to_native_inference(
            analysis['unet_output_01'], analysis['ifft_mag_320'], analysis['ifft_complex_native'])
        unet_meas = from_kspace(to_kspace(unet_native) * analysis['mask'])
        return center_crop((unet_native - unet_meas).abs().unsqueeze(0), (320, 320)).squeeze()


class HighFreqEnergyDetector(HallucinationDetector):
    """Laplacian + DoG on network additions. Based on NB03 high-freq enrichment."""
    def name(self): return "High-Freq Energy"
    def detect(self, analysis):
        ifft = analysis['ifft_mag_320']
        unet = analysis['unet_output_01'] * (ifft.max() - ifft.min()) + ifft.min()
        diff = (unet - ifft).numpy()
        hf = np.abs(laplace(diff))
        bp = np.maximum(gaussian_filter(np.abs(diff), 2) - gaussian_filter(np.abs(diff), 8), 0)
        hf_n = hf / (hf.max() + 1e-8)
        bp_n = bp / (bp.max() + 1e-8)
        return torch.tensor(0.5 * hf_n + 0.5 * bp_n, dtype=torch.float32)


class KSpaceConsistencyDetector(HallucinationDetector):
    """Error on measured k-space lines projected back to image domain."""
    def name(self): return "K-Space Consistency"
    def detect(self, analysis):
        unet_native = lift_unet_to_native_inference(
            analysis['unet_output_01'], analysis['ifft_mag_320'], analysis['ifft_complex_native'])
        measured = to_kspace(analysis['ifft_complex_native'])
        err_k = (to_kspace(unet_native) - measured) * analysis['mask']
        return center_crop(from_kspace(err_k).abs().unsqueeze(0), (320, 320)).squeeze()


class ReferenceFreesFRC(HallucinationDetector):
    """Patch-wise FRC between U-Net and IFFT. Adapted from Kc et al. (2024)."""
    def __init__(self, patch_size=48, stride=16, frc_threshold=0.75):
        self.ps, self.stride, self.thr = patch_size, stride, frc_threshold
    def name(self): return "Ref-Free sFRC"
    def detect(self, analysis):
        ifft = analysis['ifft_mag_320'].numpy()
        unet = (analysis['unet_output_01'] * (analysis['ifft_mag_320'].max() - analysis['ifft_mag_320'].min())
                + analysis['ifft_mag_320'].min()).numpy()
        risk = np.zeros((320, 320), dtype=np.float32)
        count = np.zeros((320, 320), dtype=np.float32)
        ps = self.ps
        for i in range(0, 320-ps+1, self.stride):
            for j in range(0, 320-ps+1, self.stride):
                p1, p2 = ifft[i:i+ps, j:j+ps], unet[i:i+ps, j:j+ps]
                if p1.std() < 1e-6: continue
                hann = np.outer(np.hanning(ps), np.hanning(ps))
                f1, f2 = np.fft.fft2(p1 * hann), np.fft.fft2(p2 * hann)
                y, x = np.ogrid[-ps//2:ps//2, -ps//2:ps//2]
                r = np.fft.fftshift(np.sqrt(x**2 + y**2).astype(int))
                frc = []
                for ri in range(ps//4, ps//2):
                    ring = r == ri
                    if ring.sum() == 0: continue
                    num = np.abs(np.sum(f1[ring] * np.conj(f2[ring])))
                    den = np.sqrt(np.sum(np.abs(f1[ring])**2) * np.sum(np.abs(f2[ring])**2))
                    frc.append(num / (den + 1e-10))
                if frc:
                    score = np.mean([f < self.thr for f in frc])
                    risk[i:i+ps, j:j+ps] += score
                    count[i:i+ps, j:j+ps] += 1
        valid = count > 0
        risk[valid] /= count[valid]
        return torch.tensor(risk, dtype=torch.float32)


class MultiMaskConsensusDetector(HallucinationDetector):
    """Std across U-Net outputs from different random masks. Narnhofer-inspired."""
    def __init__(self, model, device, n_masks=8):
        self.model, self.device, self.n = model, device, n_masks
    def name(self): return f"Multi-Mask ({self.n})"
    def detect(self, analysis):
        with h5py.File(analysis['h5_path'], 'r') as f:
            kspace = torch.tensor(f['kspace'][analysis['slice_idx']].copy(), dtype=torch.complex64)
        H, W = kspace.shape
        outputs = []
        self.model.eval()
        for seed in range(self.n):
            mask = create_mask((H, W), analysis['acceleration'], 'random', seed=seed+1000)
            ifft = center_crop(from_kspace(kspace * mask).abs().unsqueeze(0), (320,320)).squeeze()
            ifft_n = (ifft - ifft.min()) / (ifft.max() - ifft.min() + 1e-8)
            with torch.no_grad():
                out = self.model(ifft_n.unsqueeze(0).unsqueeze(0).to(self.device)).squeeze().cpu().clamp(0,1)
            outputs.append(out)
        return torch.stack(outputs).std(dim=0)


# ---- Learned Risk Map (architecture) ----

class SmallUNet(nn.Module):
    """3-level U-Net for risk prediction. Input: 2ch (IFFT + PSF), Output: 1ch."""
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(2,32,3,padding=1), nn.ReLU(True), nn.Conv2d(32,32,3,padding=1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1), nn.ReLU(True), nn.Conv2d(64,64,3,padding=1), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.ReLU(True), nn.Conv2d(128,128,3,padding=1), nn.ReLU(True))
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(nn.Conv2d(128,256,3,padding=1), nn.ReLU(True), nn.Conv2d(256,256,3,padding=1), nn.ReLU(True))
        self.up3 = nn.ConvTranspose2d(256,128,2,stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(256,128,3,padding=1), nn.ReLU(True), nn.Conv2d(128,128,3,padding=1), nn.ReLU(True))
        self.up2 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(128,64,3,padding=1), nn.ReLU(True), nn.Conv2d(64,64,3,padding=1), nn.ReLU(True))
        self.up1 = nn.ConvTranspose2d(64,32,2,stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(64,32,3,padding=1), nn.ReLU(True), nn.Conv2d(32,32,3,padding=1), nn.ReLU(True))
        self.final = nn.Conv2d(32,1,1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.up3(b)
        if d3.shape != e3.shape: d3 = F.pad(d3, [0,e3.shape[3]-d3.shape[3],0,e3.shape[2]-d3.shape[2]])
        d3 = self.dec3(torch.cat([e3, d3], 1))
        d2 = self.up2(d3)
        if d2.shape != e2.shape: d2 = F.pad(d2, [0,e2.shape[3]-d2.shape[3],0,e2.shape[2]-d2.shape[2]])
        d2 = self.dec2(torch.cat([e2, d2], 1))
        d1 = self.up1(d2)
        if d1.shape != e1.shape: d1 = F.pad(d1, [0,e1.shape[3]-d1.shape[3],0,e1.shape[2]-d1.shape[2]])
        d1 = self.dec1(torch.cat([e1, d1], 1))
        return self.final(d1)


class LearnedRiskDetector(HallucinationDetector):
    """Small CNN predicting hallucination risk from IFFT + PSF."""
    def __init__(self, risk_model, device):
        self.risk_model, self.device = risk_model, device
    def name(self): return "Learned Risk"
    def detect(self, analysis):
        ifft_norm, _, _ = normalize(analysis['ifft_mag_320'])
        psf = compute_psf_1d(analysis['mask'])
        W = analysis['kspace_shape'][1]
        start = (W - 320) // 2
        psf_2d = torch.tensor(psf[start:start+320], dtype=torch.float32).unsqueeze(0).expand(320, -1)
        inp = torch.stack([ifft_norm, psf_2d], 0).unsqueeze(0).to(self.device)
        self.risk_model.eval()
        with torch.no_grad():
            return F.relu(self.risk_model(inp).squeeze().cpu())


class OracleDetector(HallucinationDetector):
    """Upper bound: the null-space map itself (requires GT)."""
    def name(self): return "Oracle (GT)"
    def detect(self, analysis):
        return analysis['unet_decomp']['null_map']


# Instantiate non-learned detectors
det_psf = PSFDetector()
det_residual = ResidualDetector()
det_dc = DataConsistencyDetector()
det_hf = HighFreqEnergyDetector()
det_kspace = KSpaceConsistencyDetector()
det_sfrc = ReferenceFreesFRC()
det_multimask = MultiMaskConsensusDetector(model, device, n_masks=8)
det_oracle = OracleDetector()

print("8 detectors defined + oracle upper bound")
print(f"SmallUNet: {sum(p.numel() for p in SmallUNet().parameters()):,} params")

## 3. Evaluation Framework

Patch-level AUROC is the primary metric. Pixel-level AUROC is inappropriate for diffuse MRI hallucinations: all detectors score 0.55 to 0.60 at pixel level because hallucinations are spread across the image as high-frequency texture, not concentrated in blobs. Patch-level evaluation (averaging risk and ground truth per patch, then computing AUROC on patch scores) captures the spatial context needed.

This aligns with the sFRC methodology (Kc et al., 2024) which evaluates at patch granularity.

In [ ]:
def binarize_ground_truth(null_map, threshold=80.0):
    cutoff = np.percentile(null_map.numpy(), threshold)
    return (null_map >= cutoff).float()

def compute_auroc(risk_map, gt_binary, subsample=4):
    r = risk_map[::subsample, ::subsample].flatten().numpy()
    g = gt_binary[::subsample, ::subsample].flatten().numpy()
    if g.sum() == 0 or g.sum() == len(g): return 0.5
    return roc_auc_score(g, r)

def compute_auroc_patches(risk_map, gt_null_map, patch_size=32):
    """Patch-level AUROC: average per patch, binarize at 80th percentile."""
    H, W = risk_map.shape
    r_np, g_np = risk_map.numpy(), gt_null_map.numpy()
    pr, pg = [], []
    for i in range(0, H-patch_size+1, patch_size):
        for j in range(0, W-patch_size+1, patch_size):
            pr.append(r_np[i:i+patch_size, j:j+patch_size].mean())
            pg.append(g_np[i:i+patch_size, j:j+patch_size].mean())
    pr, pg = np.array(pr), np.array(pg)
    thresh = np.percentile(pg, 80)
    pb = (pg >= thresh).astype(float)
    if pb.sum() == 0 or pb.sum() == len(pb): return 0.5
    return roc_auc_score(pb, pr)

def compute_fpr_at_tpr(risk_map, gt_binary, target_tpr=0.95, subsample=4):
    r = risk_map[::subsample, ::subsample].flatten().numpy()
    g = gt_binary[::subsample, ::subsample].flatten().numpy()
    if g.sum() == 0 or g.sum() == len(g): return 1.0
    fpr_arr, tpr_arr, _ = roc_curve(g, r)
    idx = min(np.searchsorted(tpr_arr, target_tpr), len(fpr_arr)-1)
    return fpr_arr[idx]

print("Evaluation framework ready (patch-level AUROC primary, pixel-level secondary)")

## 4. Generate Evaluation Dataset

Split validation volumes: first half for training the learned risk map, second half for evaluating all detectors. One middle slice per volume.

In [ ]:
h5_files = sorted(Path(local_val).glob('*.h5'))
n_train = len(h5_files) // 2
train_files = h5_files[:n_train]
test_files = h5_files[n_train:]
print(f"Train: {len(train_files)} volumes, Test: {len(test_files)} volumes")

test_analyses = []
t0 = time.time()
for i, h5_path in enumerate(test_files):
    try:
        test_analyses.append(full_analysis_slice(str(h5_path), model, device, acceleration=4.0, seed=42))
        if (i+1) % 20 == 0:
            print(f"  {i+1}/{len(test_files)} ({time.time()-t0:.0f}s)")
    except Exception as e:
        pass

print(f"\n{len(test_analyses)} test analyses in {time.time()-t0:.0f}s")
if test_analyses:
    print(f"  IFFT null: {test_analyses[0]['ifft_decomp']['null_ratio']*100:.1f}%")
    print(f"  U-Net null: {test_analyses[0]['unet_decomp']['null_ratio']*100:.1f}%")

## 5. Evaluate Non-Learned Detectors

All detectors except the learned risk map (which needs training first) and multi-mask (evaluated separately due to runtime).

In [ ]:
quick_detectors = [
    ("IFFT Gradient", det_psf),
    ("|U-Net - IFFT|", det_residual),
    ("Data Consistency", det_dc),
    ("High-Freq Energy", det_hf),
    ("K-Space Consistency", det_kspace),
    ("Ref-Free sFRC", det_sfrc),
    ("Oracle (GT)", det_oracle),
]

for ps in [16, 32]:
    print(f"\n--- Patch-Level AUROC ({ps}x{ps}, top 20%) ---")
    for name, det in quick_detectors:
        aurocs = []
        t0 = time.time()
        for a in test_analyses:
            try:
                aurocs.append(compute_auroc_patches(det.detect(a), a['unet_decomp']['null_map'], ps))
            except: pass
        aurocs = np.array(aurocs) if aurocs else np.array([0.5])
        gt = "Yes" if "Oracle" in name else "No"
        print(f"  {name:>25}: {aurocs.mean():.3f} +/- {aurocs.std():.3f}  (GT: {gt}, {time.time()-t0:.1f}s)")

## 6. Multi-Mask Consensus

The strongest single detector. Runs the U-Net with 8 different random masks on the same k-space. Per-pixel standard deviation reveals where the reconstruction is mask-dependent, which is exactly where hallucinations live (NB03 control: r drops from 0.95 to 0.66 with different masks).

In [ ]:
print("Evaluating Multi-Mask (8 masks per slice, slower)...")
t0 = time.time()
mm_aurocs_16, mm_aurocs_32 = [], []
for a in tqdm(test_analyses, desc="Multi-Mask"):
    try:
        risk = det_multimask.detect(a)
        mm_aurocs_16.append(compute_auroc_patches(risk, a['unet_decomp']['null_map'], 16))
        mm_aurocs_32.append(compute_auroc_patches(risk, a['unet_decomp']['null_map'], 32))
    except: pass

mm16 = np.array(mm_aurocs_16) if mm_aurocs_16 else np.array([0.5])
mm32 = np.array(mm_aurocs_32) if mm_aurocs_32 else np.array([0.5])
print(f"Multi-Mask (8):  16x16 = {mm16.mean():.3f} +/- {mm16.std():.3f}")
print(f"                 32x32 = {mm32.mean():.3f} +/- {mm32.std():.3f}")
print(f"  ({time.time()-t0:.0f}s)")

## 7. Train Learned Risk Map

A small U-Net (~1.9M params) that takes IFFT magnitude and the PSF map as input and predicts pixel-wise hallucination risk. Trained for 30 epochs on null-space error maps from the training split using 4 mask seeds per volume.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class RiskDataset(Dataset):
    def __init__(self, h5_files, model, device, seeds=[42, 123, 456, 789]):
        self.samples = []
        for i, h5_path in enumerate(h5_files):
            for seed in seeds:
                try:
                    a = full_analysis_slice(str(h5_path), model, device, acceleration=4.0, seed=seed)
                    ifft_n, _, _ = normalize(a['ifft_mag_320'])
                    psf = compute_psf_1d(a['mask'])
                    W = a['kspace_shape'][1]
                    psf_2d = torch.tensor(psf[(W-320)//2:(W-320)//2+320], dtype=torch.float32).unsqueeze(0).expand(320,-1)
                    null_n, _, _ = normalize(a['unet_decomp']['null_map'])
                    self.samples.append({'input': torch.stack([ifft_n, psf_2d], 0), 'target': null_n.unsqueeze(0)})
                except: pass
            if (i+1) % 25 == 0:
                print(f"  {i+1}/{len(h5_files)} ({len(self.samples)} samples)")
        print(f"  Total: {len(self.samples)} training samples")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]['input'], self.samples[idx]['target']


print("Building risk training dataset...")
risk_dataset = RiskDataset(train_files, model, device)
risk_loader = DataLoader(risk_dataset, batch_size=4, shuffle=True, num_workers=0)

risk_model = SmallUNet().to(device)
risk_opt = optim.Adam(risk_model.parameters(), lr=1e-3)
risk_sched = optim.lr_scheduler.CosineAnnealingLR(risk_opt, T_max=30)
criterion = nn.MSELoss()

history = []
for epoch in range(1, 31):
    risk_model.train()
    total_loss, n = 0.0, 0
    for inp, tgt in risk_loader:
        loss = criterion(risk_model(inp.to(device)), tgt.to(device))
        risk_opt.zero_grad(); loss.backward(); risk_opt.step()
        total_loss += loss.item(); n += 1
    risk_sched.step()
    avg = total_loss / max(1, n)
    history.append(avg)
    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:2d}: loss={avg:.6f}")

# Save
torch.save({'model_state_dict': risk_model.state_dict(), 'history': history},
           '/content/drive/MyDrive/fastmri/checkpoints/risk_map_v1.pt')
print(f"Saved risk model (final loss: {history[-1]:.6f})")

# Create detector
det_learned = LearnedRiskDetector(risk_model, device)

## 8. Full Comparison and Combinations

All detectors evaluated together, plus weighted combinations. The combination of multi-mask consensus with the learned risk map or IFFT gradient can push past individual detector limits.

In [ ]:
# Evaluate learned risk
lr_16, lr_32 = [], []
for a in test_analyses:
    try:
        risk = det_learned.detect(a)
        lr_16.append(compute_auroc_patches(risk, a['unet_decomp']['null_map'], 16))
        lr_32.append(compute_auroc_patches(risk, a['unet_decomp']['null_map'], 32))
    except: pass
lr_16, lr_32 = np.array(lr_16), np.array(lr_32)
print(f"Learned Risk:  16x16 = {lr_16.mean():.3f} +/- {lr_16.std():.3f}")
print(f"               32x32 = {lr_32.mean():.3f} +/- {lr_32.std():.3f}")


# Combinations
class CombinedDetector(HallucinationDetector):
    def __init__(self, det_a, det_b, w_a=0.5, label="Combined"):
        self.a, self.b, self.w, self.label = det_a, det_b, w_a, label
    def name(self): return self.label
    def detect(self, analysis):
        a = self.a.detect(analysis); b = self.b.detect(analysis)
        a_n = (a - a.min()) / (a.max() - a.min() + 1e-8)
        b_n = (b - b.min()) / (b.max() - b.min() + 1e-8)
        return self.w * a_n + (1-self.w) * b_n

combos = [
    ("MM + Learned (70/30)", CombinedDetector(det_multimask, det_learned, 0.7, "MM+LR 70/30")),
    ("MM + Learned (50/50)", CombinedDetector(det_multimask, det_learned, 0.5, "MM+LR 50/50")),
    ("MM + IFFT Grad (60/40)", CombinedDetector(det_multimask, det_psf, 0.6, "MM+PSF 60/40")),
]

print("\n--- Combinations (32x32 patches) ---")
for name, det in combos:
    aurocs = []
    for a in tqdm(test_analyses, desc=name, leave=False):
        try:
            aurocs.append(compute_auroc_patches(det.detect(a), a['unet_decomp']['null_map'], 32))
        except: pass
    aurocs = np.array(aurocs) if aurocs else np.array([0.5])
    print(f"  {name:>28}: {aurocs.mean():.3f} +/- {aurocs.std():.3f}")

## 9. Visualization: Detection Maps and ROC Curves

In [ ]:
# Publication figure: 3 slices x 7 columns
os.makedirs('figures', exist_ok=True)
examples = [test_analyses[0], test_analyses[len(test_analyses)//3], test_analyses[2*len(test_analyses)//3]]

fig, axes = plt.subplots(3, 7, figsize=(24, 10))
col_titles = ['Ground Truth', 'U-Net', 'Hallucination\n(GT null-space)',
              'IFFT Gradient', '|U-Net-IFFT|', 'Data Consistency', 'Learned Risk']

for row, ex in enumerate(examples):
    esc = ex['esc_target']
    unet = ex['unet_output_01'] * (esc.max()-esc.min()) + esc.min()
    null_map = ex['unet_decomp']['null_map']
    risks = [det_psf.detect(ex), det_residual.detect(ex), det_dc.detect(ex), det_learned.detect(ex)]

    axes[row,0].imshow(esc.numpy(), cmap='gray'); axes[row,0].axis('off')
    axes[row,1].imshow(unet.numpy(), cmap='gray'); axes[row,1].axis('off')
    im = axes[row,2].imshow(null_map.numpy(), cmap='hot'); axes[row,2].axis('off')
    plt.colorbar(im, ax=axes[row,2], fraction=0.046)

    for j, risk in enumerate(risks):
        im = axes[row,3+j].imshow(risk.numpy(), cmap='hot'); axes[row,3+j].axis('off')
        plt.colorbar(im, ax=axes[row,3+j], fraction=0.046)

    axes[row,0].set_ylabel(f'Vol {row+1}', fontsize=12, rotation=0, labelpad=40, va='center')

for j, t in enumerate(col_titles):
    axes[0,j].set_title(t, fontsize=11)

plt.suptitle('Physics-Informed Hallucination Detection (all GT-free at inference)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/04_publication_figure.png', bbox_inches='tight')
plt.show()


# ROC curves
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
colors = {'PSF': '#e41a1c', 'Residual': '#377eb8', 'DC': '#4daf4a', 'Learned': '#984ea3'}

for key, det, color in [('PSF', det_psf, '#e41a1c'), ('Residual', det_residual, '#377eb8'),
                          ('DC', det_dc, '#4daf4a'), ('Learned', det_learned, '#984ea3')]:
    all_r, all_g = [], []
    for a in test_analyses:
        gt = binarize_ground_truth(a['unet_decomp']['null_map'], 80)
        all_g.append(gt[::4,::4].flatten().numpy())
        all_r.append(det.detect(a)[::4,::4].flatten().numpy())
    fpr, tpr, _ = roc_curve(np.concatenate(all_g), np.concatenate(all_r))
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{key} (AUROC={auc(fpr,tpr):.3f})')

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves (top 20% null-space threshold)')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/04_roc_curves.png', bbox_inches='tight')
plt.show()

![Figure 1](../figures/figures_notebook_4/fig_001.png?raw=1)


![Figure 2](../figures/figures_notebook_4/fig_002.png?raw=1)


In [ ]:
det_pairs = [('PSF', det_psf), ('Residual', det_residual), ('DC', det_dc), ('Learned', det_learned)]
n_d = len(det_pairs)
corr_matrix = np.zeros((n_d, n_d))

for a in test_analyses[:30]:
    risks = [det.detect(a).flatten().numpy() for _, det in det_pairs]
    for i in range(n_d):
        for j in range(n_d):
            corr_matrix[i,j] += pearsonr(risks[i], risks[j])[0]
corr_matrix /= min(30, len(test_analyses))

fig, ax = plt.subplots(figsize=(6, 5))
names = [n for n, _ in det_pairs]
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n_d)); ax.set_yticks(range(n_d))
ax.set_xticklabels(names, rotation=30, ha='right'); ax.set_yticklabels(names)
for i in range(n_d):
    for j in range(n_d):
        ax.text(j, i, f'{corr_matrix[i,j]:.2f}', ha='center', va='center',
                color='white' if abs(corr_matrix[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, label='Pearson r')
ax.set_title('Detector Complementarity')
plt.tight_layout()
plt.savefig('figures/04_complementarity.png', bbox_inches='tight')
plt.show()

![Figure 3](../figures/figures_notebook_4/fig_003.png?raw=1)


## 10. Save Results

In [ ]:
nb04_results = {
    'best_detector': 'Learned Risk',
    'ranking_32x32': {
        'Learned Risk': 0.891,
        'MM + Learned (50/50)': 0.888,
        'MM + Learned (70/30)': 0.864,
        'IFFT Gradient': 0.827,
        'MM + IFFT Grad (60/40)': 0.822,
        'Multi-Mask (8)': 0.807,
        'High-Freq Energy': 0.711,
        '|U-Net - IFFT|': 0.709,
        'K-Space Consistency': 0.704,
        'Data Consistency': 0.675,
        'Ref-Free sFRC': 0.532,
        'Oracle (GT)': 1.000,
    },
    'pixel_level_roc': {
        'PSF': 0.525, 'Residual': 0.546, 'DC': 0.538, 'Learned': 0.622,
    },
    'complementarity': {
        'PSF-Residual': 0.23, 'PSF-DC': 0.28, 'PSF-Learned': 0.44,
        'Residual-DC': 0.18, 'Residual-Learned': 0.31, 'DC-Learned': 0.32,
    },
    'n_test_slices': len(test_analyses),
}

os.makedirs('/content/drive/MyDrive/fastmri/checkpoints', exist_ok=True)
with open('/content/drive/MyDrive/fastmri/checkpoints/nb04_results_final.json', 'w') as f:
    json.dump(nb04_results, f, indent=2)
!cp figures/04_*.png /content/drive/MyDrive/fastmri/checkpoints/ 2>/dev/null || true
print("Results and figures saved to Drive.")

## Summary

| Detector | AUROC (32x32) | Cost | GT-free? |
|----------|--------------|------|----------|
| Learned Risk | 0.891 | 1 small CNN | Yes |
| MM + Learned (50/50) | 0.888 | 8 passes + 1 CNN | Yes |
| MM + Learned (70/30) | 0.864 | 8 passes + 1 CNN | Yes |
| IFFT Gradient | 0.827 | Negligible | Yes |
| MM + IFFT Grad (60/40) | 0.822 | 8 passes + Sobel | Yes |
| Multi-Mask (8) | 0.807 | 8 forward passes | Yes |
| High-Freq Energy | 0.711 | Negligible | Yes |
| |U-Net - IFFT| | 0.709 | Negligible | Yes |
| K-Space Consistency | 0.704 | 1 FFT | Yes |
| Data Consistency | 0.675 | 1 FFT | Yes |
| Ref-Free sFRC | 0.532 | High | Yes |
| Oracle (GT needed) | 1.000 | n/a | No |

The learned risk map (0.891) is the strongest single detector by taking both IFFT magnitude (anatomy) and the PSF map (physics) as input. This makes sense: NB03 showed r = 0.95 physics contribution plus r = 0.66 anatomy contribution, and the learned model captures both.

IFFT gradient (0.827) is the best zero-cost detector, requiring only a Sobel filter on the already-available IFFT image. No learning, no ground truth, no extra forward passes.

Multi-mask consensus (0.807) is weaker than expected individually, but combinations with the learned risk map nearly match the learned detector alone. The pixel-level ROC confirms that all detectors operate near chance at pixel granularity (0.52 to 0.62), validating the use of patch-level evaluation.

The low cross-correlations between detectors (0.18 to 0.44) confirm complementarity, which Notebook 05 will exploit by adding model-level uncertainty methods (MC Dropout, Deep Ensembles).

**Next:** Notebook 05 adds uncertainty-based detectors and the full 22-detector benchmark.